# Cloud-Base Aerosol–Cloud Susceptibility (Global NorESM)

**Scientific question:** At *cloud base* — the level where activation actually happens — how do the key aerosol–cloud quantities relate globally? Specifically: how susceptible is cloud droplet number (CDNC) to CCN proxies (N20/N70/N100), and when CDNC, maximum supersaturation (Smax) and sub-grid updraft (Wsub) are considered together, which controls the droplet-competition signal once updraft is held fixed?

Every field is first reduced from `(time, lev, lat, lon)` to a single cloud-base level per column (the surface-most valid level), giving `(time, lat, lon)`. All correlations and susceptibility slopes are then computed per grid cell over time, in log space.

**Inputs** (all on `/share/pech2273/`): `combined.nc` (CDNC), `smax_ratios/WSUB.nc`, `smax_ratios/smax_ratios.nc`, and the CCN-proxy number concentrations `NorESM_CDNC_CCN/N20|N70|N100.nc`.
**Outputs:** global maps of median cloud-base fields, per-cell correlation and partial-correlation maps, and binned-median susceptibility maps per CCN proxy.

The cloud-base sampling helpers (`cloud_base_index`, `at_cloud_base`), the correlation/partial-correlation map builders (`corr_map`, `partial_corr`, `partial_corr_map`), and the susceptibility builder (`susceptibility_map`, `binned_slope`) now live in `PeterChurchillFunctions`.

## 1 · Setup and imports

In [ ]:
# --- Imports ---
import PeterChurchillFunctions as Function
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from scipy.stats import rankdata

# --- Configuration (single source of truth) ---
SHARE = "/share/pech2273"
CDNC_PATH = f"{SHARE}/combined.nc"
WSUB_PATH = f"{SHARE}/smax_ratios/WSUB.nc"
SMAX_PATH = f"{SHARE}/smax_ratios/smax_ratios.nc"
CCN_PATHS = {
    'N20':  f"{SHARE}/NorESM_CDNC_CCN/N20.nc",
    'N70':  f"{SHARE}/NorESM_CDNC_CCN/N70.nc",
    'N100': f"{SHARE}/NorESM_CDNC_CCN/N100.nc",
}

# Minimum joint-valid timesteps a cell needs to enter a correlation/partial map.
MIN_N = 50

## 2 · Load fields and sample at cloud base

`combined.nc` provides CDNC, whose finite-data footprint defines the cloud-base level per column (`cloud_base_index`). Every other field is then sampled at that same level with `at_cloud_base`, yielding `(time, lat, lon)` arrays that are NaN wherever a column has no cloud.

In [ ]:
Updraft = xr.open_dataset(WSUB_PATH)
CDNC = xr.open_dataset(CDNC_PATH, chunks={}).CDNC
smax = xr.open_dataset(SMAX_PATH)

# Cloud-base level index from CDNC's valid footprint.
idx, has_cloud = Function.cloud_base_index(CDNC)

# Sample the core fields at cloud base.
CDNC_CB = Function.at_cloud_base(CDNC, idx, has_cloud)
Updraft_CB = Function.at_cloud_base(Updraft.WSUB, idx, has_cloud)

# Smax: take the per-column max of the in-cloud and cloud-volume ratios.
smax_incld_CB = Function.at_cloud_base(smax.incld_ratio, idx, has_cloud)
smax_cldv_CB = Function.at_cloud_base(smax.cldv_ratio, idx, has_cloud)
smax_CB = np.maximum(smax_incld_CB, smax_cldv_CB)

In [ ]:
# CCN proxies at cloud base.
ccn_ds = {name: xr.open_dataset(path) for name, path in CCN_PATHS.items()}
N20_CB = Function.at_cloud_base(ccn_ds['N20'].N20, idx, has_cloud)
N70_CB = Function.at_cloud_base(ccn_ds['N70'].N70, idx, has_cloud)
N100_CB = Function.at_cloud_base(ccn_ds['N100'].N100, idx, has_cloud)
CCN_CB = {'N20': N20_CB, 'N70': N70_CB, 'N100': N100_CB}

## 3 · Median cloud-base fields

Time-median of each cloud-base field per grid cell — the climatological picture of updraft, Smax and CDNC at the activation level.

In [ ]:
median_fields = [
    (Updraft_CB.median('time', skipna=True), 'Median cloud-base updraft (m s$^{-1}$)', 'viridis'),
    (smax_CB.median('time', skipna=True),    'Median maximum supersaturation',          'viridis'),
    (CDNC_CB.median('time', skipna=True),     'Median CDNC [# m$^{-3}$]',                'viridis'),
]

fig, axes = plt.subplots(3, 1, figsize=(11, 13),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (field, label, cmap) in zip(axes, median_fields):
    im = ax.pcolormesh(field['lon'], field['lat'], field.values,
                       cmap=cmap, transform=ccrs.PlateCarree(), shading='auto')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_global()
    ax.set_title(label)
    plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8, label=label)
plt.tight_layout()
plt.show()

## 4 · Global relationships between the three drivers

Pool every valid cloud-base point and look at the bulk correlation structure of Smax, CDNC and updraft, raw and log-transformed (the fields span orders of magnitude, so logs linearise the multiplicative relationships). Spearman is included as a rank-based, nonlinearity-robust check.

In [ ]:
s = smax_CB.values.ravel()
c = CDNC_CB.values.ravel()
u = Updraft_CB.values.ravel()

m = np.isfinite(s) & np.isfinite(c) & np.isfinite(u)
df = pd.DataFrame({'Smax': s[m], 'CDNC': c[m], 'Updraft': u[m]})

print("Pearson (raw):\n", df.corr())
logdf = np.log(df.where(df > 0)).dropna()
print("\nPearson (log):\n", logdf.corr())
print("\nSpearman (rank):\n", df.corr(method='spearman'))

### Global partial correlations

With all three in log space, the first-order partials separate the direct Smax–CDNC link from the part mediated by updraft. `Function.partial_corr` applies the standard formula to the 3×3 correlation matrix; the rank version is the Spearman analogue.

In [ ]:
ls, lc, lu = logdf['Smax'].values, logdf['CDNC'].values, logdf['Updraft'].values
R = np.corrcoef(np.vstack([ls, lc, lu]))   # 0=Smax, 1=CDNC, 2=Updraft

print("Pearson partials:")
print("  Smax–CDNC | Updraft :", Function.partial_corr(R, 0, 1, 2))
print("  Smax–Updraft | CDNC :", Function.partial_corr(R, 0, 2, 1))
print("  CDNC–Updraft | Smax :", Function.partial_corr(R, 1, 2, 0))

Rr = np.corrcoef(np.vstack([rankdata(ls), rankdata(lc), rankdata(lu)]))
print("\nSpearman partials:")
print("  Smax–CDNC | Updraft :", Function.partial_corr(Rr, 0, 1, 2))
print("  Smax–Updraft | CDNC :", Function.partial_corr(Rr, 0, 2, 1))
print("  CDNC–Updraft | Smax :", Function.partial_corr(Rr, 1, 2, 0))

## 5 · Per-cell correlation maps

Where (geographically) does each pairwise relationship hold? `Function.corr_map` returns the per-cell Pearson correlation over time, masking cells with fewer than `MIN_N` joint-valid timesteps.

In [ ]:
r_smax_cdnc = Function.corr_map(smax_CB, CDNC_CB, min_n=MIN_N)
r_smax_updraft = Function.corr_map(smax_CB, Updraft_CB, min_n=MIN_N)
r_cdnc_updraft = Function.corr_map(CDNC_CB, Updraft_CB, min_n=MIN_N)

corr_panels = [
    (r_smax_cdnc, 'Smax vs CDNC'),
    (r_smax_updraft, 'Smax vs Updraft'),
    (r_cdnc_updraft, 'CDNC vs Updraft'),
]

fig, axes = plt.subplots(3, 1, figsize=(10, 13),
                         subplot_kw={'projection': ccrs.Robinson()})
for ax, (r, title) in zip(axes, corr_panels):
    im = ax.pcolormesh(r['lon'], r['lat'], r.values, cmap='RdBu_r',
                       vmin=-1, vmax=1, transform=ccrs.PlateCarree(), shading='auto')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_global()
    ax.set_title(title)
    plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8, label='Pearson r')
plt.tight_layout()
plt.show()

### CCN proxy vs Smax maps

The same per-cell correlation, this time between Smax and each CCN proxy — more aerosol should suppress peak supersaturation (competition), so these are expected to skew negative.

In [ ]:
r_smax_ccn = {name: Function.corr_map(smax_CB, ccn_cb, min_n=MIN_N)
              for name, ccn_cb in CCN_CB.items()}

fig, axes = plt.subplots(3, 1, figsize=(10, 13),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, name in zip(axes, ['N20', 'N70', 'N100']):
    r = r_smax_ccn[name]
    im = ax.pcolormesh(r['lon'], r['lat'], r.values, cmap='RdBu_r',
                       vmin=-1, vmax=1, transform=ccrs.PlateCarree(), shading='auto')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_global()
    ax.set_title(f'Smax vs {name}')
    plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8, label='Pearson r')
plt.tight_layout()
plt.show()

## 6 · Per-cell partial correlations (droplet competition, updraft controlled)

The headline diagnostic: per cell, the partial correlation of CDNC with Smax controlling for updraft, and of CDNC with updraft controlling for Smax. `Function.partial_corr_map` does the per-cell log-space fit and masking. If Smax–CDNC dominates once updraft is fixed, the first map should be strongly signed where the second is weak.

In [ ]:
pr_cdnc_smax = Function.partial_corr_map(CDNC_CB, smax_CB, Updraft_CB, min_n=MIN_N)
pr_cdnc_updr = Function.partial_corr_map(CDNC_CB, Updraft_CB, smax_CB, min_n=MIN_N)

fig, axes = plt.subplots(2, 1, figsize=(11, 9),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, dm, title in [(axes[0], pr_cdnc_smax, 'CDNC–Smax | Updraft'),
                      (axes[1], pr_cdnc_updr, 'CDNC–Updraft | Smax')]:
    im = ax.pcolormesh(dm['lon'], dm['lat'], dm.values, cmap='RdBu_r',
                       vmin=-1, vmax=1, transform=ccrs.PlateCarree(), shading='auto')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_global()
    ax.set_title(title)
    plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8, label='partial r')
plt.tight_layout()
plt.show()

w = np.cos(np.deg2rad(pr_cdnc_smax['lat']))
print('CDNC–Smax|Updraft  area-wt mean:', float(pr_cdnc_smax.weighted(w).mean()))
print('CDNC–Updraft|Smax  area-wt mean:', float(pr_cdnc_updr.weighted(w).mean()))

### Distribution of the per-cell partials

Histogram of the two partial-correlation maps across grid cells, with medians marked. This summarises how consistently the Smax pathway dominates the updraft pathway cell-by-cell.

In [ ]:
a = pr_cdnc_smax.values.ravel()
b = pr_cdnc_updr.values.ravel()
a = a[np.isfinite(a)]
b = b[np.isfinite(b)]

fig, ax = plt.subplots(figsize=(8, 5))
bins = np.linspace(-1, 1, 61)
ax.hist(a, bins=bins, alpha=0.6, label='CDNC–Smax | Updraft', color='steelblue')
ax.hist(b, bins=bins, alpha=0.6, label='CDNC–Updraft | Smax', color='indianred')
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.axvline(np.median(a), color='steelblue', lw=1.5, ls=':')
ax.axvline(np.median(b), color='indianred', lw=1.5, ls=':')
ax.set_xlabel('partial correlation r')
ax.set_ylabel('number of grid cells')
ax.set_title('Distribution of per-cell partial correlations')
ax.legend()
plt.tight_layout()
plt.show()

print(f"CDNC–Smax|Updraft : median {np.median(a):+.3f}, mean {a.mean():+.3f}, n={a.size}")
print(f"CDNC–Updraft|Smax : median {np.median(b):+.3f}, mean {b.mean():+.3f}, n={b.size}")
print(f"  cells where Smax-partial < 0: {100*(a<0).mean():.1f}%")
print(f"  cells where Updraft-partial > 0: {100*(b>0).mean():.1f}%")

## 7 · Joint structure: Smax / updraft / CDNC

A pooled 2-D view of the triangle. The hexbin colours each (log Smax, log CDNC) cell by the mean updraft within it, exposing how updraft organises the Smax–CDNC plane.

In [ ]:
s = np.asarray(smax_CB.values).ravel()
c = np.asarray(CDNC_CB.values).ravel()
w = np.asarray(Updraft_CB.values).ravel()

m = np.isfinite(s) & np.isfinite(c) & np.isfinite(w) & (s > 0) & (c > 0) & (w > 0)
ls = np.log(s[m])
lc = np.log10(c[m])
wv = w[m]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
hb = ax.hexbin(ls, lc, C=wv, reduce_C_function=np.mean, gridsize=60,
               cmap="viridis", norm=LogNorm(), mincnt=5)
fig.colorbar(hb, ax=ax, label="mean updraft w (m/s)")
ax.set_xlabel("ln(S$_{max}$)")
ax.set_ylabel("log10(CDNC)  [m$^{-3}$]")
ax.set_title("ln(S$_{max}$) vs CDNC, coloured by mean updraft (log scale)")
plt.tight_layout()
plt.show()

## 8 · Cloud-base susceptibility maps by CCN proxy

The aerosol–cloud susceptibility d ln(CDNC) / d ln(CCN), per grid cell, via the binned-median log-log slope (`Function.susceptibility_map`). Computed for each of N20, N70, N100. Arid/no-data cells render grey.

*(This is the expensive cell — three full per-cell loops over the globe.)*

In [ ]:
suscept = {}
for name, ccn_cb in CCN_CB.items():
    print(f'computing susceptibility vs {name}...')
    suscept[name] = Function.susceptibility_map(ccn_cb, CDNC_CB)
    s_map = suscept[name]
    w = np.cos(np.deg2rad(s_map['lat']))
    awm = float(s_map.where(np.isfinite(s_map)).weighted(w).mean())
    print(f'  valid {int(np.isfinite(s_map).sum())}/{s_map.size}, area-wt mean {awm:.3f}')

In [ ]:
import copy
cmap = copy.copy(plt.cm.viridis)
cmap.set_bad(alpha=0)

fig, axes = plt.subplots(3, 1, figsize=(11, 13),
                         subplot_kw={'projection': ccrs.PlateCarree()})
im = None
for ax, name in zip(axes, ['N20', 'N70', 'N100']):
    s_map = suscept[name]
    ax.set_facecolor('lightgrey')
    ax.add_feature(cfeature.LAND, facecolor='lightgrey', zorder=0)
    im = ax.pcolormesh(s_map['lon'], s_map['lat'], s_map.values, cmap=cmap,
                       vmin=0, vmax=1, transform=ccrs.PlateCarree(),
                       shading='auto', zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2)
    ax.set_global()
    w = np.cos(np.deg2rad(s_map['lat']))
    awm = float(s_map.where(np.isfinite(s_map)).weighted(w).mean())
    ax.set_title(f'CDNC susceptibility vs {name}  (area-wt mean {awm:.3f})')

cb = fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.7, pad=0.02)
cb.set_label('dln(CDNC) / dln(CCN)')
fig.suptitle('Cloud-base aerosol–cloud susceptibility by CCN proxy', y=0.995)
plt.show()

### What does susceptibility track?

Relate the N70 susceptibility map to the time-median cloud-base fields (CDNC, Smax, updraft) and to the temporal variability of the CCN proxy. High susceptibility is expected where CCN is the limiting factor (clean, CCN-variable regimes) rather than updraft-limited.

In [ ]:
cdnc_med = CDNC_CB.median('time', skipna=True)
smax_med = smax_CB.median('time', skipna=True)
updr_med = Updraft_CB.median('time', skipna=True)
susc = suscept['N70']

df = pd.DataFrame({
    'Susceptibility': susc.values.ravel(),
    'CDNC': cdnc_med.values.ravel(),
    'Smax': smax_med.values.ravel(),
    'Updraft': updr_med.values.ravel(),
}).replace([np.inf, -np.inf], np.nan).dropna()

print("Spearman (susceptibility vs median fields):")
print(df[['Susceptibility', 'CDNC', 'Smax', 'Updraft']].corr(method='spearman')['Susceptibility'])

# Susceptibility vs temporal variability of the CCN proxy.
ccn_logstd = np.log(N70_CB.where(N70_CB > 0)).std('time')
d = pd.DataFrame({'susc': susc.values.ravel(),
                  'logCCN_std': ccn_logstd.values.ravel()}).dropna()
print('\nSpearman susceptibility vs CCN log-variability:', d.corr(method='spearman').iloc[0, 1])

## 9 · Single-cell drill-down

Sanity-check the susceptibility map at one location (SE Pacific stratocumulus deck) by reconstructing the binned-median fit from that cell's raw time series and overlaying it on the joint N70–CDNC histogram.

In [ ]:
target_lat, target_lon = -20.0, 270.0
ia = int(np.abs(CDNC_CB['lat'].values - target_lat).argmin())
ib = int(np.abs(CDNC_CB['lon'].values - target_lon).argmin())
print(f"cell: lat {float(CDNC_CB['lat'][ia]):.1f}, lon {float(CDNC_CB['lon'][ib]):.1f}")

x_raw = N70_CB.values[:, ia, ib]
y_raw = CDNC_CB.values[:, ia, ib]
m = np.isfinite(x_raw) & np.isfinite(y_raw) & (x_raw > 0) & (y_raw > 0)
x = np.log(x_raw[m])
y = np.log(y_raw[m])
print(f"valid timesteps: {x.size}")

# Binned-median points + slope, reusing the module's bin settings.
order = np.argsort(x)
xs, ys = x[order], y[order]
mx, my = [], []
for chunk in np.array_split(np.arange(xs.size), 20):
    if chunk.size >= 4:
        mx.append(np.median(xs[chunk]))
        my.append(np.median(ys[chunk]))
mx, my = np.array(mx), np.array(my)
slope, intercept = np.polyfit(mx, my, 1)
print(f"binned-median slope (this cell): {slope:.3f}")
print(f"map value for cross-check:       {float(suscept['N70'][ia, ib]):.3f}")

fig, ax = plt.subplots(figsize=(7.5, 6))
hb = ax.hexbin(x, y, gridsize=40, cmap='Blues', mincnt=1)
fig.colorbar(hb, ax=ax, label='count')
ax.plot(mx, my, 'o', color='orange', ms=8, mec='k', label='bin medians')
xline = np.linspace(x.min(), x.max(), 50)
ax.plot(xline, slope * xline + intercept, 'r-', lw=2, label=f'slope = {slope:.3f}')
ax.set_xlabel('ln(N70)  [m$^{-3}$]')
ax.set_ylabel('ln(CDNC)  [m$^{-3}$]')
ax.set_title(f'N70 vs CDNC at cell ({float(CDNC_CB["lat"][ia]):.1f}, {float(CDNC_CB["lon"][ib]):.1f})')
ax.legend()
plt.tight_layout()
plt.show()